In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib as plt
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
import keras
from keras.utils import to_categorical
import os

In [9]:
import joblib

In [3]:
print(os.getcwd())

/Users/shivaram/network-anomaly-detection-ids/Notebooks


In [4]:
print(tf.__version__)

2.21.0


In [5]:
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes',
    'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

df_train = pd.read_csv("../data/KDDTrain+.txt", header = None, names = columns)
df_test = pd.read_csv("../data/KDDTest+.txt", header = None, names = columns)

df_train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [6]:
print(df_train.shape)
print(df_train.columns.tolist())

(125973, 43)
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']


In [7]:
print(df_train.isnull().sum().sum())

0


In [8]:
print(df_train['label'].unique())

<StringArray>
[         'normal',         'neptune',     'warezclient',         'ipsweep',
       'portsweep',        'teardrop',            'nmap',           'satan',
           'smurf',             'pod',            'back',    'guess_passwd',
       'ftp_write',        'multihop',         'rootkit', 'buffer_overflow',
            'imap',     'warezmaster',             'phf',            'land',
      'loadmodule',             'spy',            'perl']
Length: 23, dtype: str


In [9]:
print(df_train['protocol_type'].unique())

<StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str


In [10]:
print(df_train['service'].unique())

<StringArray>
[   'ftp_data',       'other',     'private',        'http',  'remote_job',
        'name',  'netbios_ns',       'eco_i',         'mtp',      'telnet',
      'finger',    'domain_u',      'supdup',   'uucp_path',      'Z39_50',
        'smtp',    'csnet_ns',        'uucp', 'netbios_dgm',       'urp_i',
        'auth',      'domain',         'ftp',         'bgp',        'ldap',
       'ecr_i',      'gopher',       'vmnet',      'systat',    'http_443',
         'efs',       'whois',       'imap4',    'iso_tsap',        'echo',
      'klogin',        'link',      'sunrpc',       'login',      'kshell',
     'sql_net',        'time',   'hostnames',        'exec',       'ntp_u',
     'discard',        'nntp',     'courier',         'ctf',         'ssh',
     'daytime',       'shell',     'netstat',       'pop_3',        'nnsp',
         'IRC',       'pop_2',     'printer',       'tim_i',     'pm_dump',
       'red_i', 'netbios_ssn',         'rje',         'X11',       'urh_i'

In [11]:
print(df_train['flag'].unique())

<StringArray>
['SF', 'S0', 'REJ', 'RSTR', 'SH', 'RSTO', 'S1', 'RSTOS0', 'S3', 'S2', 'OTH']
Length: 11, dtype: str


In [12]:
x = df_train.iloc[:, :- 2].values
y = df_train.iloc[:, -2]

x.shape

(125973, 41)

In [10]:
def pre_process():
    x = df_train.iloc[:, :- 2].values
    y = df_train['label'].values

    # get the columns with non numerical values in x
    categorical_cols = [1, 2, 3]
    numerical_cols = [i for i in range(x.shape[1]) if i not in categorical_cols]
    
    # use column transformer from sklearn
    """
    we are converting the strings in categorical columns using one hot encoder and scaling the values in
    numerical columns using MinMaxScaler
    
    """
    ct = ColumnTransformer(
        transformers = [
            ('cat', OneHotEncoder(), categorical_cols),
            ('num',MinMaxScaler(), numerical_cols)
        ]
    )
    
    X = ct.fit_transform(x)

    # use label encoder to encode the values of y 
    le = LabelEncoder()

    y_encoded = le.fit_transform(y)

    Y = to_categorical(y_encoded)

    return X, Y, ct, le
    

In [11]:
X, Y, ct, le = pre_process()

# for the auto encoder we need the values of x with only label == normal
normal_mask = df_train['label'] == 'normal'

# get only the normal ones as X_normal
X_normal = X[normal_mask]
Y_normal = Y[normal_mask]
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"X_normal shape: {X_normal.shape}")
print(f"Y_normal shape: {Y_normal.shape}")

X shape: (125973, 122)
Y shape: (125973, 23)
X_normal shape: (67343, 122)
Y_normal shape: (67343, 23)


In [12]:
# save X, Y, X_normal
np.save('../data/X_train.npy', X)
np.save('../data/X_normal.npy', X_normal)
np.save('../data/Y_train.npy', Y)
np.save('../data/Y_normal.npy', Y_normal)

# save the files needed to preprocess the Test dataset.
joblib.dump(ct, '../data/column_transformer.pkl')
joblib.dump(le, '../data/label_encoder.pkl')

print("files saved sucessfully")

files saved sucessfully


In [13]:
# see if any test labels are new
train_labels = set(df_train['label'].unique())
test_labels = set(df_test['label'].unique())
print("Labels in test but not train:", test_labels - train_labels)

Labels in test but not train: {'named', 'xlock', 'apache2', 'mscan', 'ps', 'snmpguess', 'worm', 'saint', 'snmpgetattack', 'processtable', 'xterm', 'httptunnel', 'mailbomb', 'sendmail', 'udpstorm', 'xsnoop', 'sqlattack'}


In [15]:
print(len(le.classes_))

23


In [19]:
# As we see above that that there new labels in test that were not in train
known_labels = df_test['label'].isin(set(le.classes_)) # the 23 labels in train
df_test_known = df_test[known_labels] # only the labels in both test and train


#load transoformers
ct = joblib.load('../data/column_transformer.pkl')
le = joblib.load('../data/label_encoder.pkl')

# create X_test and Y_test
x_test = df_test_known.iloc[:, : -2].values
y_test = df_test_known['label'].values

X_test = ct.transform(x_test)
Y_test_encoded = le.transform(y_test)
Y_test = to_categorical(Y_test_encoded)

np.save('../data/X_test.npy', X_test)
np.save('../data/Y_test.npy', Y_test)

print(X_test.shape)
print(Y_test.shape)

print('Test files saved sucessfully ')

(18794, 122)
(18794, 23)
Test files saved sucessfully 
